# 04 Agent01 Clean Run Preparation

Este notebook prepara una corrida limpia nueva de `quotes`. Su objetivo no es auditar el run viejo, sino construir de forma controlada y reproducible los inputs de una descarga nueva:

- universo efectivo del run
- `ticker,date` usando ventana temporal explícita
- recorte por lifecycle
- calendario operativo de días de mercado
- artefactos de entrada del run
- comando final sugerido para Agent01

Regla de diseño:
- este notebook prepara inputs
- no descarga datos
- no valida datos
- no hace coverage
- no hace recovery

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
QUOTES_ROOT = Path(r"D:\quotes")

# Run nuevo limpio: este run reutiliza D:\quotes con --resume y solo debe descargar lo no presente.
NEW_RUN_ID = "20260319_quotes_clean_v2_draft"
RUN_DIR = PROJECT_ROOT / "runs" / "polygon_realtime_audit" / NEW_RUN_ID
INPUTS_DIR = RUN_DIR / "inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Inputs oficiales del builder
UNIVERSE_REFINED = PROJECT_ROOT / "data" / "reference" / "universe_pti" / "tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet"
OFFICIAL_LIFECYCLE = PROJECT_ROOT / "data" / "reference" / "official_lifecycle_compiled.csv"

# Ventana explícita del run: estas fechas sí son política del Task Builder.
RUN_DATE_FROM = pd.Timestamp("2005-01-01")
RUN_DATE_TO = pd.Timestamp("2026-12-31")
CALENDAR_POLICY = "business_day_intersection_with_run_window"  # v2 temporal mientras el calendario oficial no esté inyectado aquí

# Artefactos de salida del run limpio.
UNIVERSE_RUN_CSV = INPUTS_DIR / "tickers_quotes_prod_v2_clean.csv"
TASKS_RUN_CSV = INPUTS_DIR / "tasks_quotes_prod_v2_clean.csv"
TASKS_RUN_META_JSON = INPUTS_DIR / "tasks_quotes_prod_v2_clean.meta.json"

context = {
    "project_root": str(PROJECT_ROOT),
    "new_run_id": NEW_RUN_ID,
    "run_dir": str(RUN_DIR),
    "quotes_root": str(QUOTES_ROOT),
    "universe_refined": str(UNIVERSE_REFINED),
    "official_lifecycle": str(OFFICIAL_LIFECYCLE),
    "run_date_from": str(RUN_DATE_FROM.date()),
    "run_date_to": str(RUN_DATE_TO.date()),
    "calendar_policy": CALENDAR_POLICY,
    "resume_policy": "resume_over_existing_disk_at_quotes_root",
}
print(json.dumps(context, indent=2, ensure_ascii=False))

{
  "project_root": "C:\\TSIS_Data\\v1\\backtest_SmallCaps",
  "new_run_id": "20260319_quotes_clean_v2_draft",
  "run_dir": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260319_quotes_clean_v2_draft",
  "quotes_root": "D:\\quotes",
  "universe_refined": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
  "official_lifecycle": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
  "run_date_from": "2005-01-01",
  "run_date_to": "2026-12-31",
  "calendar_policy": "business_day_intersection_with_run_window",
  "resume_policy": "resume_over_existing_disk_at_quotes_root"
}


**universo upstream para este run limpio será**  
- tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet  

**lifecycle oficial será**  
- official_lifecycle_compiled.csv  

**documentado qué política temporal se está usando:**  
- por ahora business_day_intersection_with_run_window
- más adelante debería ser calendario oficial XNYS

In [2]:
from pathlib import Path

SCRIPT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\quotes\04_universe_lifecycle_snapshot.py"
    )
exec(SCRIPT.read_text(encoding="utf-8"), globals())

{
  "universe_refined": {
    "path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
    "exists": true,
    "size_mb": 0.08,
    "rows": 12133,
    "unique_tickers": 12133,
    "columns": [
      "ticker"
    ]
  },
  "official_lifecycle": {
    "path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
    "exists": true,
    "size_mb": 0.08,
    "rows": 1970,
    "unique_tickers": 1969,
    "columns": [
      "ticker",
      "cik",
      "list_date",
      "delist_date",
      "evidence_count",
      "source_count",
      "source_doc_types"
    ],
    "list_date_min": "1995-09-13",
    "list_date_max": "2026-02-18",
    "delist_date_min": "2006-02-23",
    "delist_date_max": "2026-02-27"
  },
  "cross_check": {
    "universe_tickers": 12133,
    "lifecycle_tickers": 1970,
    "intersection_tickers": 1961,
    "universe_not_in_lifecycle":

,ticker
0,AAA
1,AABA
2,AAC
3,AACB
4,AACI
5,AACQ
6,AACT
7,AAGR
8,AAI
9,AAIC



Muestra official_lifecycle:


,ticker,cik,list_date,delist_date,evidence_count,source_count,source_doc_types
0,AABA,1011006,2017-06-16,2019-10-02,3,2,Exchange Notice|Form 8-K
1,AACI,2092897,2026-02-17,NaT,1,1,8-A12B
2,AAT,1500217,2011-01-11,NaT,1,1,8-A12B
3,ABAT,1576873,2013-10-17,NaT,1,1,8-A12G
4,ABCL,1703057,2020-12-08,NaT,1,1,8-A12B
5,ABEO,318306,2014-11-04,2019-12-20,2,1,25-NSE|8-A12B
6,ABLV,1957489,2023-08-17,NaT,1,1,8-A12B
7,ABOS,1576885,2021-06-28,NaT,1,1,8-A12B
8,ABP,1893219,2022-01-13,2024-11-12,2,1,25-NSE|8-A12B
9,ABR,1253986,2013-02-01,2021-06-24,2,1,25-NSE|8-A12B



Sample universe_not_in_lifecycle:


,ticker
0,AAA
1,AAC
2,AACB
3,AACQ
4,AACT
5,AAGR
6,AAI
7,AAIC
8,AAL
9,AAM



Sample lifecycle_not_in_universe:


,ticker
0,AHH
1,BEAT
2,BLBX
3,ETHZ
4,GMGI
5,GOLD
6,NITO
7,VRAR


Esta celda:

* toma el universo refinado de quotes, lo cruza con lifecycle oficial, recorta cada ticker a la ventana temporal
* válida del run, elimina casos inválidos y deja la lista final de tickers que sí pueden entrar en la corrida limpia.

In [3]:
u = pd.read_parquet(UNIVERSE_REFINED) # qué tickers queremos
lc = pd.read_csv(OFFICIAL_LIFECYCLE) # cuándo estuvieron vivos oficialmente

# normalizamos
u["ticker"] = u["ticker"].astype(str).str.upper().str.strip() 
lc["ticker"] = lc["ticker"].astype(str).str.upper().str.strip() 

lc["list_date"] = pd.to_datetime(lc["list_date"], errors="coerce")
lc["delist_date"] = pd.to_datetime(lc["delist_date"], errors="coerce")

# Universo efectivo del run: tickers del universo refinado con lifecycle oficial utilizable.
base = lc[lc["ticker"].isin(set(u["ticker"]))].copy() #  Cruza universo refinado con lifecycle
base["run_start"] = base["list_date"].clip(lower=RUN_DATE_FROM) # ventana temporal del run para cada ticker
base["run_end"] = base["delist_date"].fillna(RUN_DATE_TO).clip(upper=RUN_DATE_TO)
base = base.dropna(subset=["ticker", "run_start", "run_end"]).copy()
base = base[base["run_end"] >= base["run_start"]].copy()
base = base.sort_values(["ticker", "run_start"]).drop_duplicates(subset=["ticker"], keep="first").reset_index(drop=True)

snapshot = {
    "universe_refined_rows": int(len(u)),
    "universe_refined_unique_tickers": int(u['ticker'].nunique()),
    "official_lifecycle_rows": int(len(lc)),
    "effective_run_tickers": int(base['ticker'].nunique()),
    "effective_min_start": str(base['run_start'].min().date()) if len(base) else None,
    "effective_max_end": str(base['run_end'].max().date()) if len(base) else None,
}
print(json.dumps(snapshot, indent=2, ensure_ascii=False))
base.head(10)

{
  "universe_refined_rows": 12133,
  "universe_refined_unique_tickers": 12133,
  "official_lifecycle_rows": 1970,
  "effective_run_tickers": 1961,
  "effective_min_start": "2005-01-01",
  "effective_max_end": "2026-12-31"
}


,ticker,cik,list_date,delist_date,evidence_count,source_count,source_doc_types,run_start,run_end
0,AABA,1011006,2017-06-16,2019-10-02,3,2,Exchange Notice|Form 8-K,2017-06-16,2019-10-02
1,AACI,2092897,2026-02-17,NaT,1,1,8-A12B,2026-02-17,2026-12-31
2,AAT,1500217,2011-01-11,NaT,1,1,8-A12B,2011-01-11,2026-12-31
3,ABAT,1576873,2013-10-17,NaT,1,1,8-A12G,2013-10-17,2026-12-31
4,ABCL,1703057,2020-12-08,NaT,1,1,8-A12B,2020-12-08,2026-12-31
5,ABEO,318306,2014-11-04,2019-12-20,2,1,25-NSE|8-A12B,2014-11-04,2019-12-20
6,ABLV,1957489,2023-08-17,NaT,1,1,8-A12B,2023-08-17,2026-12-31
7,ABOS,1576885,2021-06-28,NaT,1,1,8-A12B,2021-06-28,2026-12-31
8,ABP,1893219,2022-01-13,2024-11-12,2,1,25-NSE|8-A12B,2022-01-13,2024-11-12
9,ABR,1253986,2013-02-01,2021-06-24,2,1,25-NSE|8-A12B,2013-02-01,2021-06-24


## Política temporal de v2

En el run antiguo las tasks se generaron con `pd.bdate_range(list_date, end_date)` sin recorte explícito de la ventana del run. Aquí se aplica una política mínima v2:

- `run_start = max(list_date, RUN_DATE_FROM)`
- `run_end = min(delist_date or RUN_DATE_TO, RUN_DATE_TO)`
- si `run_end < run_start`, el ticker se excluye

Nota: esta versión sigue usando `bdate_range` como aproximación operativa. El siguiente endurecimiento debe ser cambiarlo por calendario oficial XNYS.

In [4]:
rows = []
for _, r in base.iterrows():
    days = pd.bdate_range(r["run_start"].date(), r["run_end"].date())
    rows.extend({"ticker": r["ticker"], "date": d.strftime("%Y-%m-%d")} for d in days)

tasks = pd.DataFrame(rows)
tasks = tasks.drop_duplicates(subset=["ticker", "date"]).sort_values(["ticker", "date"]).reset_index(drop=True)

task_audit = {
    "tasks_rows": int(len(tasks)),
    "tasks_unique_tickers": int(tasks['ticker'].nunique()) if len(tasks) else 0,
    "tasks_date_min": str(pd.to_datetime(tasks['date']).min().date()) if len(tasks) else None,
    "tasks_date_max": str(pd.to_datetime(tasks['date']).max().date()) if len(tasks) else None,
    "pre_2005_rows": int((pd.to_datetime(tasks['date']) < RUN_DATE_FROM).sum()) if len(tasks) else 0,
    "post_run_end_rows": int((pd.to_datetime(tasks['date']) > RUN_DATE_TO).sum()) if len(tasks) else 0,
    "duplicates_task_key": int(tasks.duplicated(['ticker', 'date']).sum()) if len(tasks) else 0,
}
print(json.dumps(task_audit, indent=2, ensure_ascii=False))
tasks.head(10)

{
  "tasks_rows": 3304899,
  "tasks_unique_tickers": 1961,
  "tasks_date_min": "2005-01-03",
  "tasks_date_max": "2026-12-31",
  "pre_2005_rows": 0,
  "post_run_end_rows": 0,
  "duplicates_task_key": 0
}


,ticker,date
0,AABA,2017-06-16
1,AABA,2017-06-19
2,AABA,2017-06-20
3,AABA,2017-06-21
4,AABA,2017-06-22
5,AABA,2017-06-23
6,AABA,2017-06-26
7,AABA,2017-06-27
8,AABA,2017-06-28
9,AABA,2017-06-29


In [5]:
# Persistencia del run limpio.
preferred_cols = [
    "ticker",
    "cik",
    "list_date",
    "delist_date",
    "evidence_count",
    "source_count",
    "source_doc_types",
    "run_start",
    "run_end",
]
available_cols = [c for c in preferred_cols if c in base.columns]
base_to_save = base[available_cols].copy()
base_to_save.to_csv(UNIVERSE_RUN_CSV, index=False)
tasks.to_csv(TASKS_RUN_CSV, index=False)

meta = {
    "run_id": NEW_RUN_ID,
    "source_universe_refined": str(UNIVERSE_REFINED),
    "source_official_lifecycle": str(OFFICIAL_LIFECYCLE),
    "calendar_policy": CALENDAR_POLICY,
    "run_date_from": str(RUN_DATE_FROM.date()),
    "run_date_to": str(RUN_DATE_TO.date()),
    "tickers_count": int(base_to_save['ticker'].nunique()),
    "tasks_total": int(len(tasks)),
    "date_min": str(pd.to_datetime(tasks['date']).min().date()) if len(tasks) else None,
    "date_max": str(pd.to_datetime(tasks['date']).max().date()) if len(tasks) else None,
    "notes": [
        "v2 clean preparation for quotes",
        "resume over existing D:\\quotes disk; download only missing files",
        "window explicitly clipped to run_date_from/run_date_to",
        "next hardening step: replace business_day calendar with official XNYS calendar"
    ],
}
TASKS_RUN_META_JSON.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')

print(str(UNIVERSE_RUN_CSV))
print(str(TASKS_RUN_CSV))
print(str(TASKS_RUN_META_JSON))
print(json.dumps(meta, indent=2, ensure_ascii=False))

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tickers_quotes_prod_v2_clean.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tasks_quotes_prod_v2_clean.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tasks_quotes_prod_v2_clean.meta.json
{
  "run_id": "20260319_quotes_clean_v2_draft",
  "source_universe_refined": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
  "source_official_lifecycle": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
  "calendar_policy": "business_day_intersection_with_run_window",
  "run_date_from": "2005-01-01",
  "run_date_to": "2026-12-31",
  "tickers_count": 1961,
  "tasks_total": 3304899,
  "date_min": "2005-01-03",
  "date_max": "2026-12-31",
  "no

In [6]:
agent01_cmd = (
    f'python {PROJECT_ROOT / "scripts" / "download_quotes.py"} '
    f'--csv {TASKS_RUN_CSV} '
    f'--output {QUOTES_ROOT} '
    f'--concurrent 32 '
    f'--run-id {NEW_RUN_ID} '
    f'--run-dir {RUN_DIR} '
    f'--resume '
    f'--task-batch-size 1000'
)
print(agent01_cmd)

python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py --csv C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tasks_quotes_prod_v2_clean.csv --output D:\quotes --concurrent 32 --run-id 20260319_quotes_clean_v2_draft --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft --resume --task-batch-size 1000


```sh
PS C:\Users\AlexJ> python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py --csv C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tasks_quotes_prod_v2_clean.csv --output D:\quotes --concurrent 32 --run-id 20260319_quotes_clean_v2_draft --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft --resume --task-batch-size 1000
run_id=20260319_quotes_clean_v2_draft
run_dir=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft
output_root=D:\quotes
csv=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft\inputs\tasks_quotes_prod_v2_clean.csv
tasks_total=3304899
tasks_already_ok=0
tasks_to_process=3304899
processed=100/3304899
processed=200/3304899
processed=300/3304899
```

**PROBLEMA: tarderamos 3 dias más cuando ya hay 85% descargado**  
**SOLUCION:**  

He creado la lanzadera aquí:

- `build_quotes_missing_only_from_disk.ps1`

Qué hace:

- parte de tasks_quotes_prod_v2_clean.csv
- para cada ticker,date calcula el path esperado en D:\quotes
- solo da por “ya existente” un file si:
    - existe
    - el parquet es legible
    - num_rows > 0
- además, si el file aparece como HARD_FAIL en Agent02 viejo, lo fuerza a redescarga aunque exista

Eso es lo más conservador que puedes hacer sin volver a descargar todo.

Lánzalo así:

```sh
powershell -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\build_quotes_missing_only_from_disk.ps1 -SourceRunId 20260319_quotes_clean_v2_draft -QuotesRoot D:\quotes -Agent02RunId 20260313_quotes_prod_full_12133_clean -ProgressEvery 25000
```
Te dejará:

- tasks_quotes_prod_v2_clean_missing_only.csv
- tasks_quotes_prod_v2_clean_existing_ok.csv
- tasks_quotes_prod_v2_clean_disk_audit.csv
- tasks_quotes_prod_v2_clean_missing_only.meta.json

Y al final imprime el comando nuevo de Agent01 sobre missing_only.

Mi recomendación operativa:

- para el run actual de 3.3M
- lanza esta auditoría primero
- cuando termine, para el downloader largo
- relanza Agent01 con el CSV missing_only


**Qué debes esperar al final**

Lo importante del resumen será:

- tasks_total
- existing_ok_skip
- missing_only_download
- missing_on_disk
- existing_zero_rows
- existing_unreadable
- agent02_hard_fail_redownload

La cifra crítica es:

- missing_only_download

Si esa sale muy por debajo de 3,304,899, habrás recuperado el tiempo de verdad.

**Qué haría después**

Cuando termine:

1. revisa el JSON meta
2. si missing_only_download tiene un tamaño razonable
3. relanza con el comando que imprimirá el propio script

```SH
PS C:\Users\AlexJ> powershell -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification_v2\cell_code\build_quotes_missing_only_from_disk.ps1 -SourceRunId 20260319_quotes_clean_v2_draft -QuotesRoot D:\quotes -Agent02RunId 20260313_quotes_prod_full_12133_clean -ProgressEvery 25000
Construyendo tasks missing_only para quotes...
  source_run_id : 20260319_quotes_clean_v2_draft
  source_run_dir: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260319_quotes_clean_v2_draft
  quotes_root   : D:\quotes
  agent02_run   : 20260313_quotes_prod_full_12133_clean
  hard_fail     : True
  progress      : cada 25000 tareas
{"stage": "build_missing_only_from_disk", "processed_tasks": 25000, "tasks_total": 3304899, "pct": 0.76, "last_task_key": "ACIC|2014-07-02", "rate_tasks_per_sec": 4927.1, "eta_minutes": 11.08, "eta_hours": 0.18}
{"stage": "build_missing_only_from_disk", "processed_tasks": 50000, "tasks_total": 3304899, "pct": 1.51, "last_task_key": "ADCT|2023-12-28", "rate_tasks_per_sec": 4769.54, "eta_minutes": 11.37, "eta_hours": 0.19}
{"stage": "build_missing_only_from_disk", "processed_tasks": 75000, "tasks_total": 3304899, "pct": 2.27, "last_task_key": "AGRO|2011-04-27", "rate_tasks_per_sec": 4782.8, "eta_minutes": 11.25, "eta_hours": 0.19}
{"stage": "build_missing_only_from_disk", "processed_tasks": 100000, "tasks_total": 3304899, "pct": 3.03, "last_task_key": "AKAN|2022-10-24", "rate_tasks_per_sec": 972.21, "eta_minutes": 54.93, "eta_hours": 0.92}
{"stage": "build_missing_only_from_disk", "processed_tasks": 125000, "tasks_total": 3304899, "pct": 3.78, "last_task_key": "ALLT|2024-12-10", "rate_tasks_per_sec": 148.02, "eta_minutes": 358.03, "eta_hours": 5.97}
{"stage": "build_missing_only_from_disk", "processed_tasks": 150000, "tasks_total": 3304899, "pct": 4.54, "last_task_key": "AMST|2025-01-03", "rate_tasks_per_sec": 121.19, "eta_minutes": 433.87, "eta_hours": 7.23}
{"stage": "build_missing_only_from_disk", "processed_tasks": 175000, "tasks_total": 3304899, "pct": 5.3, "last_task_key": "ANTA|2025-11-20", "rate_tasks_per_sec": 91.13, "eta_minutes": 572.4, "eta_hours": 9.54}

```



while ($true) { Get-Content C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\download_events_history.csv -Tail 10; Write-Host ('-'*100); Start-Sleep 5 }

while ($true) { python -c "import pandas as pd; from pathlib import Path; p=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\download_events_current.csv'); df=pd.read_csv(p, usecols=['ticker','date','status','error','processed_at_utc']);df['processed_at_utc']=pd.to_datetime(df['processed_at_utc'], errors='coerce');
print(df.sort_values('processed_at_utc').tail(1).to_string(index=False)); print('-'*80)"; Start-Sleep 5 }

while ($true) { python -c "import pandas as pd; from pathlib import Path; p=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\download_events_current.csv'); df=pd.read_csv(p, usecols=['ticker','date','status','error','processed_at_utc']); df['processed_at_utc']=pd.to_datetime(df['processed_at_utc'], errors='coerce'); df=df[df['processed_at_utc'].notna()]; print(df.sort_values('processed_at_utc').tail(10).to_string(index=False)); print('-'*100)"; Start-Sleep 5 }

while ($true) { python -c "import pandas as pd; from pathlib import Path; p=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\download_events_current.csv'); df=pd.read_csv(p, usecols=['ticker','date','status','error','processed_at_utc']);df['processed_at_utc']=pd.to_datetime(df['processed_at_utc'], errors='coerce'); df=df[df['processed_at_utc'].notna()];
print('ULTIMAS 10 GLOBALES'); print(df.sort_values('processed_at_utc').tail(10).to_string(index=False)); print();
vet=df[df['ticker']=='VET'].sort_values('processed_at_utc').tail(10); print('ULTIMAS 10 DE VET');
print(vet.to_string(index=False) if len(vet) else 'sin filas VET'); print('-'*100)"; Start-Sleep 5 }